<a href="https://colab.research.google.com/github/ardianita/data-science-2026/blob/main/Pertemuan_6_Ardianita_Fauziyah_250401020128.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd, seaborn as sns
import matplotlib.pyplot as plt
df = sns.load_dataset('titanic')
# Pilih kolom yang akan digunakan
cols = [
    'pclass',
    'sex',
    'age',
    'sibsp',
    'parch',
    'fare',
    'embarked',
    'survived'
]

df = df[cols].copy()
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDistribusi target:')
print(df['survived'].value_counts(normalize=True).round(3))
df

Shape: (891, 8)

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi target:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0
...,...,...,...,...,...,...,...,...
886,2,male,27.0,0,0,13.0000,S,0
887,1,female,19.0,0,0,30.0000,S,1
888,3,female,NaN,1,2,23.4500,S,0
889,1,male,26.0,0,0,30.0000,C,1


In [2]:
# Handling Missing Value
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
print('Missing setelah handling:')
print(df.isnull().sum())

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


In [3]:
# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(
    df,
    columns=['sex', 'embarked'],
    drop_first=True, # hindari dummy variable trap
    dtype=int
)
print('Kolom setelah encoding:')
print(df.columns.tolist())
df

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


,pclass,age,sibsp,parch,fare,survived,sex_male,embarked_Q,embarked_S
0,3,22.0,1,0,7.2500,0,1,0,1
1,1,38.0,1,0,71.2833,1,0,0,0
2,3,26.0,0,0,7.9250,1,0,0,1
3,1,35.0,1,0,53.1000,1,0,0,1
4,3,35.0,0,0,8.0500,0,1,0,1
...,...,...,...,...,...,...,...,...,...
886,2,27.0,0,0,13.0000,0,1,0,1
887,1,19.0,0,0,30.0000,1,0,0,1
888,3,28.0,1,2,23.4500,0,0,0,1
889,1,26.0,0,0,30.0000,1,1,0,0


In [4]:
from sklearn.model_selection import train_test_split

# Drop kelas 'survived' menjaga proporsi kelas 'survived'
x = df.drop('survived', axis=1)

# target yang dijaga
y = df['survived']

x_training, x_test, y_training, y_test = train_test_split(
    x,
    y,
    test_size=0.2,      # proporsi 20% testing, 80% training
    random_state=42,
    stratify=y          # proporsi kelas terjaga
)
print('--- Output ---')
print(f'Training: {x_training.shape[0]} baris')
print(f'Test: {x_test.shape[0]} baris')

print('\n--- Proporsi survived di Training ---')
print(y_training.value_counts(normalize=True).round(3))

print('\n--- Proporsi survived di Test ---')
print(y_test.value_counts(normalize=True).round(3))

--- Output ---
Training: 712 baris
Test: 179 baris

--- Proporsi survived di Training ---
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

--- Proporsi survived di Test ---
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


In [6]:
from sklearn.preprocessing import StandardScaler

# Pilih kolom numerik
num_cols = [
    'pclass',
    'age',
    'sibsp',
    'parch',
    'fare'
]

# Membuat object scaler
scaler = StandardScaler()

# fit_transform pada training set (belajar mean dan std dari training)
x_training[num_cols] = scaler.fit_transform(x_training[num_cols])

# transform saja pada test set (gunakan mean dan std dari training)
x_test[num_cols] = scaler.transform(x_test[num_cols])

print('Mean scaler (dari training):', scaler.mean_.round(2))
print('Std scaler (dari training):', scaler.scale_.round(2))

print('\n---- Contoh x_training setelah scaling ----')
print(x_training.head().round(3))

print('\n---- Data siap dilatih model Machine Learning ----')
print(f'x_training: {x_training.shape}, y_training: {y_training.shape}')

Mean scaler (dari training): [-0.  0. -0. -0. -0.]
Std scaler (dari training): [1. 1. 1. 1. 1.]

---- Contoh x_training setelah scaling ----
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1
855   0.830 -0.880 -0.465  0.728 -0.468         0           0           1
801  -0.371  0.118  0.478  0.728 -0.116         0           0           1

---- Data siap dilatih model Machine Learning ----
x_training: (712, 8), y_training: (712,)
